# 02 - Feature Engineering

Starts from the cleaned dataset saved in notebook 01. No cleaning is repeated here.

Adds the columns the analysis needs: revenue per line, and date parts for
trend and seasonality charts.

In [1]:
import pandas as pd

df = pd.read_parquet("../data/processed/retail_clean.parquet")
df.shape

(1021128, 9)

In [2]:
df.dtypes

Invoice                    str
StockCode                  str
Description                str
Quantity                 int64
InvoiceDate     datetime64[us]
Price                  float64
Customer ID              Int64
Country                    str
is_cancelled              bool
dtype: object

In [3]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_cancelled
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False


## Revenue

`Revenue = Quantity × Price` - the single most important derived column. Every
revenue figure in the EDA, SQL and dashboard traces back to this line.

Cancelled invoices carry negative quantities, so their revenue is naturally
negative. That is intentional: it makes lost revenue measurable instead of hidden.

In [4]:
df["Revenue"] = df["Quantity"] * df["Price"]
df[["Quantity", "Price", "Revenue"]].head()

,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [5]:
df.groupby("is_cancelled")["Revenue"].sum().round(2)

is_cancelled
False    19642692.15
True      -716425.97
Name: Revenue, dtype: float64

## Date parts

`InvoiceDate` is a single timestamp. Charts need it broken into the pieces
people actually ask questions about: which month, which weekday, which hour.

`year_month` is stored as text ("2009-12") rather than a pandas Period so it
survives the trip to parquet, PostgreSQL and Power BI unchanged. The
zero-padded format still sorts correctly.

In [6]:
df["year"] = df["InvoiceDate"].dt.year
df["month"] = df["InvoiceDate"].dt.month
df["year_month"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["day_of_week"] = df["InvoiceDate"].dt.day_name()
df["hour"] = df["InvoiceDate"].dt.hour

In [7]:
df[["InvoiceDate", "year", "month", "year_month", "day_of_week", "hour"]].head()

,InvoiceDate,year,month,year_month,day_of_week,hour
0,2009-12-01 07:45:00,2009,12,2009-12,Tuesday,7
1,2009-12-01 07:45:00,2009,12,2009-12,Tuesday,7
2,2009-12-01 07:45:00,2009,12,2009-12,Tuesday,7
3,2009-12-01 07:45:00,2009,12,2009-12,Tuesday,7
4,2009-12-01 07:45:00,2009,12,2009-12,Tuesday,7


In [8]:
df["year_month"].nunique(), df["year_month"].min(), df["year_month"].max()

(25, '2009-12', '2011-12')

In [9]:
df["day_of_week"].value_counts()

day_of_week
Thursday     193919
Tuesday      189354
Monday       181463
Wednesday    175412
Friday       147903
Sunday       132677
Saturday        400
Name: count, dtype: int64

### summary

| New column | Derived from | Used for |
|---|---|---|
| `Revenue` | `Quantity × Price` | All revenue analysis |
| `year`, `month` | `InvoiceDate` | Seasonality |
| `year_month` | `InvoiceDate` | Monthly trend, cohorts |
| `day_of_week`, `hour` | `InvoiceDate` | Behavioural patterns |

Reconciliation: non-cancelled revenue = £19,642,692.15, matching notebook 01.
Cancellations = −£716,425.97 across 17,914 rows.

Note: Saturday has only 400 rows vs 130k–190k for other weekdays - the
retailer does not trade on Saturdays. Not a data error.

## Order-level table

One row per invoice. This is the grain most business questions actually live at -
average order value, basket size, orders per customer.

Cancelled invoices are excluded: this table describes purchases, and a cancellation
is a reversal, not a purchase. They remain in `df` with negative revenue and are
analysed separately in the EDA notebook.

In [10]:
df.groupby("Invoice")["is_cancelled"].nunique().max()

1

In [11]:
sales = df[~df["is_cancelled"]]

orders = sales.groupby("Invoice").agg(
    order_date=("InvoiceDate", "min"),
    customer_id=("Customer ID", "first"),
    country=("Country", "first"),
    order_value=("Revenue", "sum"),
    n_items=("StockCode", "nunique"),
    n_units=("Quantity", "sum"),
).reset_index()

orders.shape

(39516, 7)

In [12]:
orders["order_value"].sum().round(2)

np.float64(19642692.15)

## Customer-level table

One row per customer, built from `orders` rather than from `df` - so `n_orders`
counts real orders, not line items.

Orders with no `Customer ID` are dropped here (2,922 orders, £2,574,124 -
13.1% of revenue). They stay in `df` and `orders` for revenue totals; they are
excluded only where customer identity is the unit of analysis.

In [13]:
customers = orders.dropna(subset=["customer_id"]).groupby("customer_id").agg(
    first_purchase=("order_date", "min"),
    last_purchase=("order_date", "max"),
    n_orders=("Invoice", "nunique"),
    total_revenue=("order_value", "sum"),
    country=("country", "first"),
).reset_index()

customers["aov"] = (customers["total_revenue"] / customers["n_orders"]).round(2)

customers.shape

(5852, 7)

In [14]:
customers.head()

,customer_id,first_purchase,last_purchase,n_orders,total_revenue,country,aov
0,12346,2010-03-02 13:08:00,2011-01-18 10:01:00,3,77352.96,United Kingdom,25784.32
1,12347,2010-10-31 14:20:00,2011-12-07 15:52:00,8,4921.53,Iceland,615.19
2,12348,2010-09-27 14:59:00,2011-09-25 13:13:00,5,1658.40,Finland,331.68
3,12349,2010-04-29 13:20:00,2011-11-21 09:51:00,3,3678.69,Italy,1226.23
4,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,1,294.40,Norway,294.40


In [15]:
customers[["n_orders", "total_revenue", "aov"]].describe().round(2)

,n_orders,total_revenue,aov
count,5852.00,5852.00,5852.00
mean,6.25,2916.71,384.40
std,12.75,14306.85,1254.73
min,1.00,2.95,2.95
25%,1.00,339.58,176.14
50%,3.00,856.02,277.87
75%,7.00,2241.03,410.56
max,373.00,580987.04,84236.25


In [16]:
customers["total_revenue"].sum().round(2)

np.float64(17068567.97)

### summary

| Table | Grain | Rows |
|---|---|---|
| `df` | Product line | 1,021,128 |
| `orders` | Invoice | 39,516 |
| `customers` | Customer | 5,852 |

- `orders` excludes cancellations; reconciles to £19,642,692.15
- `customers` excludes orders with no `Customer ID`; reconciles to £17,068,567.97
  (difference £2,574,124.18 = the 13.1% unattributed revenue)
- 5,852 not 5,875 customers: 23 appear only on cancelled invoices
- Value is heavily skewed - median £856 vs top customer £580,987 (680×).
  Justifies log-transform before K-Means.
- 1,618 customers (27.6%) ordered exactly once
- Caveat: 12 customers span multiple countries; `country` takes the first

## Cohort columns

A cohort is the set of customers whose first purchase fell in the same month.
`cohort_month` is stamped on the customer; `cohort_index` counts months from that
first purchase to each subsequent order.

Cohort analysis needs identified customers, so this table covers the orders
that carry a `Customer ID`. The fullorder table is unchanged and still
used for revenue totals.

In [17]:
customers["cohort_month"] = customers["first_purchase"].dt.to_period("M").astype(str)

cohorts = orders.dropna(subset=["customer_id"]).merge(
    customers[["customer_id", "first_purchase", "cohort_month"]],
    on="customer_id",
    how="left",
)

cohorts["order_month"] = cohorts["order_date"].dt.to_period("M").astype(str)

cohorts["cohort_index"] = (
    (cohorts["order_date"].dt.year - cohorts["first_purchase"].dt.year) * 12
    + (cohorts["order_date"].dt.month - cohorts["first_purchase"].dt.month)
)

cohorts.shape

(36594, 11)

In [18]:
cohorts["cohort_index"].min(), cohorts["cohort_index"].max()

(0, 24)

In [19]:
customers["cohort_month"].value_counts().sort_index().head()

cohort_month
2009-12    951
2010-01    368
2010-02    375
2010-03    441
2010-04    294
Name: count, dtype: int64

In [20]:
cohorts.head()

,Invoice,order_date,customer_id,country,order_value,n_items,n_units,first_purchase,cohort_month,order_month,cohort_index
0,489434,2009-12-01 07:45:00,13085,United Kingdom,505.30,8,166,2009-12-01 07:45:00,2009-12,2009-12,0
1,489435,2009-12-01 07:46:00,13085,United Kingdom,145.80,4,60,2009-12-01 07:45:00,2009-12,2009-12,0
2,489436,2009-12-01 09:06:00,13078,United Kingdom,630.33,19,193,2009-12-01 09:06:00,2009-12,2009-12,0
3,489437,2009-12-01 09:08:00,15362,United Kingdom,310.75,23,145,2009-12-01 09:08:00,2009-12,2009-12,0
4,489438,2009-12-01 09:24:00,18102,United Kingdom,2286.24,17,826,2009-12-01 09:24:00,2009-12,2009-12,0


## RFM base table

Recency, Frequency, Monetary - one row per customer, the input to both the
rule-based segmentation and K-Means.

**Snapshot date = last order date + 1 day = 2011-12-10**, not `today()`.
The dataset ends in December 2011, so measuring recency against the real
current date would report every customer as inactive for ~15 years, and the
numbers would change on every rerun. A fixed snapshot makes the analysis
reproducible: rerun it next year and you get identical results.

The +1 day means the most recent customer scores recency = 1, not 0 -
avoids a zero that would break the log-transform before K-Means.

In [21]:
snapshot_date = orders["order_date"].max() + pd.Timedelta(days=1)
snapshot_date

Timestamp('2011-12-10 12:50:00')

In [22]:
customer_rfm = pd.DataFrame({
    "customer_id": customers["customer_id"],
    "recency": (snapshot_date - customers["last_purchase"]).dt.days,
    "frequency": customers["n_orders"],
    "monetary": customers["total_revenue"].round(2),
})

customer_rfm.shape

(5852, 4)

In [23]:
customer_rfm.head()

,customer_id,recency,frequency,monetary
0,12346,326,3,77352.96
1,12347,2,8,4921.53
2,12348,75,5,1658.40
3,12349,19,3,3678.69
4,12350,310,1,294.40


In [24]:
customer_rfm[["recency", "frequency", "monetary"]].describe().round(2)

,recency,frequency,monetary
count,5852.00,5852.00,5852.00
mean,200.20,6.25,2916.71
std,208.51,12.75,14306.85
min,1.00,1.00,2.95
25%,25.00,1.00,339.58
50%,95.00,3.00,856.02
75%,379.00,7.00,2241.03
max,739.00,373.00,580987.04


In [25]:
customer_rfm["recency"].min(), customer_rfm["monetary"].sum().round(2)

(1, np.float64(17068567.97))

## Save the feature tables

Five tables saved as Parquet, one for each grain the later notebooks need:

| File | Grain | Rows |
|---|---|---|
| `retail_features` | Product line | 1,021,128 |
| `orders` | Invoice | 39,516 |
| `customers` | Customer | 5,852 |
| `cohorts` | Invoice (identified) | 36,594 |
| `customer_rfm` | Customer | 5,852 |

Parquet again, for the same reason as notebook 01: types survive the reload,
so no downstream notebook has to repeat any of this work.

In [26]:
df.to_parquet("../data/processed/retail_features.parquet", index=False)
orders.to_parquet("../data/processed/orders.parquet", index=False)
customers.to_parquet("../data/processed/customers.parquet", index=False)
cohorts.to_parquet("../data/processed/cohorts.parquet", index=False)
customer_rfm.to_parquet("../data/processed/customer_rfm.parquet", index=False)

In [27]:
check = pd.read_parquet("../data/processed/cohorts.parquet")
check.shape, check["cohort_index"].max()

((36594, 11), 24)

In [28]:
pd.Series({
    "line items (non-cancelled)": sales["Revenue"].sum(),
    "orders": orders["order_value"].sum(),
    "cohorts": cohorts["order_value"].sum(),
    "customers": customers["total_revenue"].sum(),
    "customer_rfm": customer_rfm["monetary"].sum(),
}).round(2)

line items (non-cancelled)    19642692.15
orders                        19642692.15
cohorts                       17068567.97
customers                     17068567.97
customer_rfm                  17068567.97
dtype: float64

### Summary - feature engineering complete

Five tables saved to `data/processed/`. All reconcile:

- £19,642,692.15 across `retail_features` and `orders` (all identified and
  unidentified purchases)
- £17,068,567.97 across `cohorts`, `customers` and `customer_rfm`
  (identified customers only)
- Difference £2,574,124.18 = the 13.1% unattributed revenue, unchanged since the cleaning stage

Key decisions:

- **Snapshot date 2011-12-10** (last order + 1 day), never `today()` - keeps
  recency reproducible and avoids a zero before the log-transform
- `cohort_index` uses plain integer month arithmetic, so it ports to SQL unchanged
- Minimum `cohort_index` is 0, which validates the customer-to-order join
- 25 cohorts, Dec 2009 – Dec 2011. The 2009-12 cohort is inflated (951 customers)
  because it absorbs everyone already active when the data begins - an artefact
  to note when reading the cohort retention heatmap, not a real acquisition spike